In [1]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai.chat_models.base import ChatOpenAI

from hakken_agents.config import LLMConfig, PromptConfig
from hakken_agents.graph_builder.schemas.extracted_facts import ExtractedFacts
from hakken_agents.utils.file import load_file
from hakken_agents.utils.prompt import load_prompt_template_from_config

load_dotenv()

True

In [2]:
ROOT_FOLDER = "/Users/Pablo.Sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/hakken-agents"
PROMPTS_FOLDER = f"{ROOT_FOLDER}/prompts/graph_builder"


llm_config = LLMConfig(name="openai/gpt-oss-120b", temperature=0.1)

In [3]:
system_prompt = PromptConfig(prompt_template_path=f"{PROMPTS_FOLDER}/extract_facts/system_v0.txt")
user_prompt = PromptConfig(prompt_template_path=f"{PROMPTS_FOLDER}/extract_facts/user_v0.txt")

facts_system_prompt = load_file(
    file_path=system_prompt.prompt_template_path,
    encoding=system_prompt.encoding,
)

variables_kwargs = {}


facts_user_prompt_template = load_prompt_template_from_config(user_prompt, **variables_kwargs)

2026-01-29 19:59:43.933 | INFO     | hakken_agents.utils.file:load_file:34 - Loading file: /Users/Pablo.Sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/hakken-agents/prompts/graph_builder/extract_facts/system_v0.txt
2026-01-29 19:59:43.934 | INFO     | hakken_agents.utils.file:load_file:38 - Loaded 321 characters from system_v0.txt


In [4]:
system_prompt = PromptConfig(
    prompt_template_path=f"{PROMPTS_FOLDER}/extract_entities/system_v0.txt"
)
user_prompt = PromptConfig(prompt_template_path=f"{PROMPTS_FOLDER}/extract_entities/user_v0.txt")

entities_system_prompt = load_file(
    file_path=system_prompt.prompt_template_path,
    encoding=system_prompt.encoding,
)

variables_kwargs = {}


entities_user_prompt_template = load_prompt_template_from_config(user_prompt, **variables_kwargs)

2026-01-29 19:59:44.458 | INFO     | hakken_agents.utils.file:load_file:34 - Loading file: /Users/Pablo.Sanchez2/Documents/GitHub/project_spaice_ds/packages/pip/hakken-agents/prompts/graph_builder/extract_entities/system_v0.txt
2026-01-29 19:59:44.460 | INFO     | hakken_agents.utils.file:load_file:38 - Loaded 208 characters from system_v0.txt


In [ ]:
query = """
FOXO3 (Forkhead box O3) is a member of the forkhead box O family of transcription factors and a key downstream effector of insulin/IGF-1 and PI3K–AKT signaling, where it regulates gene programs involved in oxidative stress resistance, autophagy, apoptosis, and metabolic homeostasis.
"""

query = """
For example, attenuation of IIS/AKT-mediated FOXO inhibition is associated with lifespan increases on the order of ~20–100% in canonical invertebrate aging models.
"""

query = """
A healthy adult human has 46 chromosomes in each body cell, a resting heart rate of about 60–100 beats per minute, and approximately 5 liters of blood circulating through the body.
"""

query = """
In a typical human cell, the TP53 gene spans about 20,000 base pairs and encodes a tumor suppressor protein of 393 amino acids, which can activate the transcription of dozens to hundreds of target genes in response to DNA damage.
"""

In [ ]:
from hakken_agents.graph_builder.schemas.extracted_entities import ExtractedEntities

llm = ChatOpenAI(
    model=llm_config.name,
    temperature=llm_config.temperature,
    api_key=llm_config.api_key,
    base_url=llm_config.base_url,
).with_structured_output(ExtractedEntities)

In [ ]:
messages = [
    SystemMessage(content=entities_system_prompt),
    HumanMessage(content=entities_user_prompt_template.format(content=query)),
]
extracted_entities = llm.invoke(messages)

In [ ]:
entities = extracted_entities.entities
entity_str = ""
for entity in entities:
    entity_str += f"- {entity.name} || {entity.domain}\n"

print(entity_str)

In [ ]:
llm = ChatOpenAI(
    model=llm_config.name,
    temperature=llm_config.temperature,
    api_key=llm_config.api_key,
    base_url=llm_config.base_url,
).with_structured_output(ExtractedFacts)

In [ ]:
messages = [
    SystemMessage(content=facts_system_prompt),
    HumanMessage(content=facts_user_prompt_template.format(content=query, entities=entity_str)),
]

In [ ]:
messages = [
    SystemMessage(content=facts_system_prompt),
    HumanMessage(content=facts_user_prompt_template.format(content=query, entities=entity_str)),
]
extracted_facts = llm.invoke(messages)

In [ ]:
for fact in extracted_facts.facts:
    print(fact)
    print("-" * 100)

In [ ]:
for fact in extracted_facts.facts:
    print(f"{fact.subject_name} ({fact.subject_domain}) | {fact.subject_quantity}")
    print(fact.relation.name)
    print(f"{fact.object_name} ({fact.object_domain}) | {fact.object_quantity}")
    print("-" * 50)